<a href="https://colab.research.google.com/github/debashisdotchatterjee/Symmetry-resolved-deconvolution-string/blob/main/Symmetry_resolved_deconvolution_string_full_code_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Symmetry-resolved deconvolution — corrected Google Colab launcher

**This notebook accepts EITHER code ZIP**. It inspects the archive and automatically runs the matching workflow:

- **Complete code ZIP** (`Symmetry_Resolved_Complete_Code.zip`): mathematical checks, simulations, introductory GWAS analysis and detailed GWAS analysis.
- **Motivation-only ZIP** (`GWAS_Motivation_Standalone_Code.zip`): introductory GWAS analysis **only**. This package intentionally does not contain `test_self_checks.py`.

The GWAS analysis uses authentic GIANT height summary statistics. The later *sign-masking experiment is deliberately imposed*; it is not a claim about missing allele labels in the source. The notebook does not invent an alternative dataset if GIANT cannot be downloaded.

Run the cells **top to bottom**. Choose exactly one code ZIP when prompted.

In [1]:
from google.colab import files
from pathlib import Path
import zipfile, os, sys, subprocess, glob, shutil, json

uploaded = files.upload()  # Select ONE: the COMPLETE code ZIP OR the MOTIVATION-ONLY code ZIP
archives = [str(n) for n in uploaded if str(n).lower().endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Select exactly one source-code ZIP (not a Results ZIP).')
package = archives[0]
with zipfile.ZipFile(package) as z:
    content = {Path(n).name for n in z.namelist() if not n.endswith('/')}
    if {'run_all_colab.py','test_self_checks.py'}.issubset(content):
        MODE = 'complete'
        expected = {'run_all_colab.py','test_self_checks.py','inverse_core.py','gwas_data.py','requirements.txt'}
    elif {'run_motivation_only.py','01_motivation_gwas.py'}.issubset(content):
        MODE = 'motivation'
        expected = {'run_motivation_only.py','01_motivation_gwas.py','inverse_core.py','gwas_data.py','requirements.txt'}
    else:
        raise ValueError('Unrecognized ZIP. Please upload Symmetry_Resolved_Complete_Code.zip or GWAS_Motivation_Standalone_Code.zip, not a results ZIP.')
    missing = expected - content
    if missing:
        raise ValueError('Source ZIP is incomplete; missing: '+', '.join(sorted(missing)))
    root=Path('/content/symmetry_code')
    if root.exists():
        shutil.rmtree(root)  # Prevent stale files from another package hiding a missing script
    root.mkdir(parents=True, exist_ok=True)
    z.extractall(root)

    # Find actual code root, including one optional parent folder in user-created ZIPs.
    candidates=[p.parent for p in root.rglob('requirements.txt') if p.is_file()]
    matches=[p for p in candidates if (p/'run_all_colab.py').exists() or (p/'run_motivation_only.py').exists()]
    if len(matches)!=1:
        raise RuntimeError('Could not determine a unique source-code directory within the ZIP.')
    CODE_ROOT=matches[0]
os.chdir(CODE_ROOT)
print('Package:', package)
print('Detected workflow:', MODE)
print('Code extracted to:', CODE_ROOT)
print('Running Python:', sys.executable)

Saving Symmetry_Resolved_Complete_Code.zip to Symmetry_Resolved_Complete_Code.zip
Package: Symmetry_Resolved_Complete_Code.zip
Detected workflow: complete
Code extracted to: /content/symmetry_code
Running Python: /usr/bin/python3


## Dependencies and mathematical checks

The mathematical self-check file is included **only in the complete package**. The notebook will not try to execute it for the motivation-only package.

In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
if MODE == 'complete':
    print('Running mathematical self-checks...', flush=True)
    subprocess.run([sys.executable, str(CODE_ROOT/'test_self_checks.py')], check=True, cwd=str(CODE_ROOT))
else:
    print('Motivation-only ZIP: no test_self_checks.py is expected; proceeding to the GWAS study.')

Running mathematical self-checks...


## Run the selected analysis

`standard` is recommended for the complete analysis; choose `quick` for a small preliminary run. The standalone motivation analysis ignores `PROFILE`.

If the historical GIANT download link does not work, upload the **original real GWAS file** and specify its exact path in `GWAS_LOCAL_FILE`; the scripts will not replace it with synthetic data.

In [5]:
PROFILE = 'full'          # complete ZIP only: 'quick', 'standard', or 'full'
GWAS_LOCAL_FILE = None       # Or e.g. '/content/GIANT_HEIGHT_Wood_et_al_2014_publicrelease_HapMapCeuFreq.txt.gz'
RUN_OPTIONAL_ECG = False     # complete ZIP only; off by default

if MODE == 'complete':
    cmd=[sys.executable, str(CODE_ROOT/'run_all_colab.py'), '--profile', PROFILE]
    if GWAS_LOCAL_FILE:
        cmd+=['--gwas-file', GWAS_LOCAL_FILE]
    if RUN_OPTIONAL_ECG:
        cmd+=['--ecg']
else:
    cmd=[sys.executable, str(CODE_ROOT/'run_motivation_only.py')]
    if GWAS_LOCAL_FILE:
        cmd+=['--gwas-file', GWAS_LOCAL_FILE]

print('Starting:', ' '.join(cmd), flush=True)
subprocess.run(cmd, check=True, cwd=str(CODE_ROOT))

if MODE == 'complete':
    manifest=Path('/content/symmetry_results/RUN_MANIFEST.json')
    if manifest.is_file():
        info=json.loads(manifest.read_text())
        print('Sections completed:',json.dumps(info.get('sections',{}),indent=2))
        if info.get('failure_details',{}).get('gwas'):
            print('WARNING: The authentic GWAS analysis did not run. Read REAL_DATA_REQUIRES_ATTENTION.txt in the results ZIP.')

Starting: /usr/bin/python3 /content/symmetry_code/run_all_colab.py --profile full
Sections completed: {
  "simulation": {
    "n_designs": 6,
    "sample_sizes": [
      250,
      750,
      1500,
      3500
    ],
    "repetitions": 250,
    "heterogeneity_repetitions": 350,
    "note": "Simulation results are Monte Carlo estimates, not predetermined outcomes."
  },
  "gwas_motivation": {
    "dataset_rows": 6000,
    "beta_scale": 0.006880239922960537,
    "cutoff_T": 4.334448160535117,
    "warning": "This is real GWAS data with artificial randomized sign labels. Reference is not latent truth."
  },
  "gwas_main": {
    "source_rows": 6000,
    "beta_scale": 0.006880239922960537,
    "replications_each_design": 140,
    "aligned_reference_cutoff": 4.34113712374582,
    "reference": "Fourier deconvolution of aligned observed GWAS coefficients, NOT latent truth.",
    "cautions": [
      "Only sign masking is randomized; genetic effect estimates and SEs are real.",
      "GIANT selec

## Download the output ZIP

For the complete package, results include simulations and whichever real-data analyses successfully completed. For the standalone package, they include only the introductory GWAS analysis. Check the run manifest before interpreting the numerical results.

In [6]:
if MODE == 'complete':
    results=sorted(glob.glob(f'/content/Symmetry_Resolved_Results_{PROFILE}_*.zip'))
else:
    results=sorted(glob.glob('/content/Symmetry_Motivation_GWAS_*.zip'))
if not results:
    raise FileNotFoundError('No results ZIP found; inspect the preceding analysis cell for the error.')
result_zip=max(results,key=os.path.getmtime)
print('Downloading:', result_zip)
files.download(result_zip)

Downloading: /content/Symmetry_Resolved_Results_full_20261102.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>